In [1]:
import pulp

# Datos de ejemplo
gerentes = ['G1', 'G2', 'G3']
capacidad = {'G1': 40, 'G2': 50, 'G3': 45}  # horas disponibles

ejecutivos = ['E1', 'E2', 'E3', 'E4', 'E5']
tiempo_requerido = {'E1': 20, 'E2': 15, 'E3': 25, 'E4': 30, 'E5': 10}  # horas que consume cada ejecutivo
valor = {'E1': 100, 'E2': 80, 'E3': 120, 'E4': 90, 'E5': 60}  # prioridad/valor del ejecutivo

# Zonas compatibles (ejecutivo -> gerentes en misma zona)
zonas_compatibles = {
    'E1': ['G1', 'G2'],
    'E2': ['G1', 'G3'],
    'E3': ['G2', 'G3'],
    'E4': ['G1'],
    'E5': ['G2', 'G3']
}



In [2]:
# Crear el problema de optimización
prob = pulp.LpProblem("Asignacion_Ejecutivos_Gerentes", pulp.LpMaximize)

# Variables de decisión: 1 si ejecutivo E es asignado a gerente G, 0 si no
asignacion = pulp.LpVariable.dicts("Asignar", 
                                  [(e, g) for e in ejecutivos for g in gerentes],
                                  cat='Binary')

# FUNCIÓN OBJETIVO: Maximizar el valor total de los ejecutivos asignados
prob += pulp.lpSum([valor[e] * asignacion[(e, g)] 
                   for e in ejecutivos for g in gerentes])



In [3]:
prob

Asignacion_Ejecutivos_Gerentes:
MAXIMIZE
100*Asignar_('E1',_'G1') + 100*Asignar_('E1',_'G2') + 100*Asignar_('E1',_'G3') + 80*Asignar_('E2',_'G1') + 80*Asignar_('E2',_'G2') + 80*Asignar_('E2',_'G3') + 120*Asignar_('E3',_'G1') + 120*Asignar_('E3',_'G2') + 120*Asignar_('E3',_'G3') + 90*Asignar_('E4',_'G1') + 90*Asignar_('E4',_'G2') + 90*Asignar_('E4',_'G3') + 60*Asignar_('E5',_'G1') + 60*Asignar_('E5',_'G2') + 60*Asignar_('E5',_'G3') + 0.0
VARIABLES
0 <= Asignar_('E1',_'G1') <= 1 Integer
0 <= Asignar_('E1',_'G2') <= 1 Integer
0 <= Asignar_('E1',_'G3') <= 1 Integer
0 <= Asignar_('E2',_'G1') <= 1 Integer
0 <= Asignar_('E2',_'G2') <= 1 Integer
0 <= Asignar_('E2',_'G3') <= 1 Integer
0 <= Asignar_('E3',_'G1') <= 1 Integer
0 <= Asignar_('E3',_'G2') <= 1 Integer
0 <= Asignar_('E3',_'G3') <= 1 Integer
0 <= Asignar_('E4',_'G1') <= 1 Integer
0 <= Asignar_('E4',_'G2') <= 1 Integer
0 <= Asignar_('E4',_'G3') <= 1 Integer
0 <= Asignar_('E5',_'G1') <= 1 Integer
0 <= Asignar_('E5',_'G2') <= 1 Integer
0 <

In [4]:
# RESTRICCIONES

# 1. Cada ejecutivo se asigna a máximo UN gerente
for e in ejecutivos:
    prob += pulp.lpSum([asignacion[(e, g)] for g in gerentes]) <= 1

# 2. No superar la capacidad de tiempo de cada gerente
for g in gerentes:
    prob += pulp.lpSum([tiempo_requerido[e] * asignacion[(e, g)] 
                       for e in ejecutivos]) <= capacidad[g]

# 3. Solo asignar si están en la misma zona
for e in ejecutivos:
    for g in gerentes:
        if g not in zonas_compatibles[e]:
            prob += asignacion[(e, g)] == 0

# Resolver el problema
prob.solve()



1

In [5]:
# Mostrar resultados
print(f"Estado: {pulp.LpStatus[prob.status]}")
print(f"Valor total obtenido: {pulp.value(prob.objective)}\n")

print("Asignaciones óptimas:")
for g in gerentes:
    print(f"\nGerente {g} (capacidad: {capacidad[g]}h):")
    tiempo_usado = 0
    for e in ejecutivos:
        if pulp.value(asignacion[(e, g)]) == 1:
            print(f"  - {e} (tiempo: {tiempo_requerido[e]}h, valor: {valor[e]})")
            tiempo_usado += tiempo_requerido[e]
    print(f"  Tiempo usado: {tiempo_usado}h")

print(f"\nEjecutivos no asignados:")
for e in ejecutivos:
    if sum(pulp.value(asignacion[(e, g)]) for g in gerentes) == 0:
        print(f"  - {e} (valor: {valor[e]})")

Estado: Optimal
Valor total obtenido: 450.0

Asignaciones óptimas:

Gerente G1 (capacidad: 40h):
  - E4 (tiempo: 30h, valor: 90)
  Tiempo usado: 30h

Gerente G2 (capacidad: 50h):
  - E1 (tiempo: 20h, valor: 100)
  - E3 (tiempo: 25h, valor: 120)
  Tiempo usado: 45h

Gerente G3 (capacidad: 45h):
  - E2 (tiempo: 15h, valor: 80)
  - E5 (tiempo: 10h, valor: 60)
  Tiempo usado: 25h

Ejecutivos no asignados:
